## DPDGF00001-1  -  Diplomado en Sismología

## Módulo: Observaciones sismológicas



#### Dependencias necesarias: Numpy, Pandas, Matplotlib, Pylab, Scipy

**Nota importante**: Gran parte de estos códigos han sido modificados a partir de las clases impartidas en la Escuela de Verano [ROSES](https://www.iris.edu/hq/inclass/course/roses)


---



# Cálculo de magnitud Mw

Hemos visto que la magnitud de los terremotos se ha calculado de diferentes maneras a medida que avanzan los años, algunas metodologías han perdurado y ocupado en la rutina de observatorios sismológicos en el mundo.

En este jupyter notebook compartido a través de google colab, veremos la Actividad 3 y final del Módulo 1 de Observaciones sismológicas.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import signal, fftpack

In [2]:
#se vincula con google drive, para poder importar los archivos
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# cargar datos, graficarlos. Inspección visual

# leo los datos deconvolucionados e integrados (a desplazamiento)
mycols=['t', 'amp']

#Acá tendrás que ingresar cada uno de los registros, por estación, por componente
#Aquí cambiar la ruta, de acuerdo a su google drive
dataframe=pd.read_csv('/content/drive/MyDrive/Doctorado/Diplomado/2026/Actividad_3/Colab/DatosInt/CX.MNMCX..HHE.D..deco.int.dat',
                      sep='\s+', names=mycols)

<>:9: SyntaxWarning:

invalid escape sequence '\s'

<>:9: SyntaxWarning:

invalid escape sequence '\s'

/tmp/ipykernel_7247/4221296253.py:9: SyntaxWarning:

invalid escape sequence '\s'



In [6]:
fig = px.line(data_frame=dataframe, x="t", y="amp", width=1000, height=400,
                    title='Serie de tiempo en unidades de desplazamiento',
                    labels={"t": "Tiempo (s)", "amp": "Amplitud (m)"})
fig.show()

In [8]:
'''
la data no es realmente perfecta, entonces podría tener saltos, tendencias, offsets, spikes raros, etc.
en este caso ocuparemos la función detrend tanto lineal (para quitar tendencias) como promedio (para quitar offset extraños).
Además, para calcular la FT, debo darle una serie de tiempo que tenga una señal periódica
'''

# ingresa tiempo inicial y final de onda S (t_S1, t_S2)
t_S1 = 76
t_S2 = 90

# separo los vectores de amplitudes y de tiempo del dataframe para ocuparlos
data_amplitudes = dataframe['amp']
data_tiempo = dataframe['t']

# calculamos el sampling (delta de tiempo)
dt = data_tiempo[1] - data_tiempo[0]

data_time = data_tiempo[int(t_S1/dt):int(t_S2/dt)]
data_ampl = data_amplitudes[int(t_S1/dt):int(t_S2/dt)]

#taper tukey window
taper = signal.windows.tukey(len(data_ampl), alpha=0.05, sym=True)
#taper = np.hanning(len(data_ampl))
data_ampl = data_ampl * taper
data_time = data_time

#paso valores a listas
data_time = data_time.tolist()
data_ampl = data_ampl.tolist()

In [9]:
fig = px.line(x=data_time, y=data_ampl, width=1000, height=400,
                    title='Serie de tiempo con tendencia removida',
                    labels={"x": "Tiempo (s)", "y": "Amplitud (m)"})

fig.show()

In [10]:
#defino largo data
npts = len(data_ampl)
print("número de puntos: %d" %(npts))

#defino delta tiempo
dt = data_time[1] - data_time[0]
print("dt: %.5f" %(dt))

#defino delta frecuencia
df = 1.0/(npts*dt)
print("df: %.5f" %(df))

número de puntos: 1400
dt: 0.01000
df: 0.07143


In [11]:
frec = np.arange(0,npts*df,df)
nf = int(npts/2) #no podemos pasar Nyquist
if len(frec) is not npts:
    frec = frec[1:nf]

fourier_amp = fftpack.fft(data_ampl)*dt
fftout = abs(fourier_amp)
fftout = fftout[1:nf]
#fftout_p = fftout/(1j*2*math.pi*frec) 	#integracion a velocidad (sólo cuando utilizamos un HN*, acelerómetro)

#print(fftout)
#print(frec)

In [12]:
#%grafico

fig = px.line(x=frec, y=fftout, width=700, height=500,
              labels={"x": "Frequency (Hz)", "y": "Displacement Spectrum Amplitude (ms)"})

fig.update_layout( yaxis = dict( showexponent = 'all', exponentformat = 'e', type="log"))
fig.update_layout( xaxis = dict( showexponent = 'all', exponentformat = 'e', type="log"))

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightPink')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightPink')

fig.show()

Como habíamos inferido anteriormente. Podemos estimar el momento sísmico $M_0$ a partir del Espectro de Fourier, gracias a la expresión:

\begin{eqnarray}
M_0 &=& \frac{4\pi\rho v^3\,R}{\left\langle \mathcal{R} \right\rangle \, FS} \Omega_0
\end{eqnarray}

Usando:
1. Un patrón de radiación promedio para la onda S por sobre toda la esfera focal. $\left\langle \mathcal{R} \right\rangle = 0.67$
2. El factor de superficie libre $FS = 2$
3. La densidad en la vecindad de la fuente $\rho=3.38 (gr/cc)$
4. La velocidad de onda S en la vecindad de la fuente $v=\beta = 4.78 (km/s)$
5. Distancia entre estación e hipocentro (ver en IRIS), $R=0.68°$ (pasar a km en script)

Además haciendo un poco de análisis dimensional tenemos:

In [13]:
#CONSTANTES
FS     = 2              # adimensional
R_rad  = 0.67           # adimensional
rho    = 3.38           # gr/cc
v      = 4.78           # km/s

#calculo la distancia hipocentral
d_epi = 113.7
depth = 90
R = np.sqrt( d_epi**2 + depth**2)

#ingreso omega por componente
omega_N = 1.04e-4  #estimado a partir del Plateau en el Espectro de Fourier
omega_E = 1.36e-4
omega_Z = 1.03e-4

omega = np.sqrt(((omega_N)**2+omega_E**2+omega_Z**2)/3)

#paso a unidades MKS
R   *= 1000
rho *= 1000
v   *= 1000

In [14]:
#Ecuación para obtener M0 a partir de un análisis espectral

M0 = (4*np.pi*rho*v**3 * R / R_rad * FS) * omega

print("El momento sísmico estimado es: %.2e Nm" % (M0))



Mw = 2/3 * np.log10(M0) - 6.07 #cuando M0 está en Nm

print("Magnitud de momento: Mw=%.2f" % (Mw))

El momento sísmico estimado es: 2.32e+17 Nm
Magnitud de momento: Mw=5.51
